# Experiment of sentiment of comments

This notebook is for experiment how classify the sentiment of the commments with the help of TextBlob library.

# Customer sentiment dataset

The dataset used comes from the kaggle platform and provided by the user Kundan Sagar. The details of the dataset extracted are from [this link](https://www.kaggle.com/datasets/kundanbedmutha/customer-sentiment-dataset).

# Importing packages

In [1]:
import pandas as pd # Useful for data wrangling
from textblob import TextBlob # Used for classify the sentiment

# Data collection

In [2]:
# You need provide the data, you can download it from
# https://www.kaggle.com/datasets/kundanbedmutha/customer-sentiment-dataset
path = "/content/drive/MyDrive/Customer_Sentiment.csv"

# Dataframe
df = pd.read_csv(path)

# First rows
df.head()

,customer_id,gender,age_group,region,product_category,purchase_channel,platform,customer_rating,review_text,sentiment,response_time_hours,issue_resolved,complaint_registered
0,1,male,60+,north,automobile,online,flipkart,1,very disappointed with the quality.,negative,46,yes,yes
1,2,other,46-60,central,books,online,swiggy instamart,5,fast delivery and great packaging.,positive,5,yes,no
2,3,female,36-45,east,sports,online,facebook marketplace,1,very disappointed with the quality.,negative,38,yes,yes
3,4,female,18-25,central,groceries,online,zepto,2,product stopped working after few days.,negative,16,yes,yes
4,5,female,18-25,east,electronics,online,croma,3,neutral about the quality.,neutral,15,yes,no


# Extracting sentiment

In [3]:
def getting_sentiment(text):
  """Function for extract the sentiment. It can analyze it in brute.
  >>> getting_sentiment("I like it! It's useful, ya")
  "positive"
  """

  # Getting the polarity
  blob = TextBlob(str(text))
  polarity = blob.sentiment.polarity

  # Classification
  if polarity > 0:
    return "positive"
  elif polarity < 0:
    return "negative"
  else:
    return "neutral"

# Samples
df["sentiment_blob"] = df["review_text"].apply(getting_sentiment)
df["polarity"] = df["review_text"].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df["subjectivity"] = df["review_text"].apply(lambda x: TextBlob(str(x)).sentiment.subjectivity)
df[["review_text", "sentiment_blob", "polarity", "subjectivity"]].sample(5)

,review_text,sentiment_blob,polarity,subjectivity
14338,very satisfied with the quality.,positive,0.650000,1.000000
7039,excellent product! exceeded expectations.,positive,1.000000,1.000000
22779,customer service was unhelpful.,neutral,0.000000,0.000000
21751,"product is okay, nothing special.",positive,0.428571,0.535714
12177,"amazing experience, highly recommend!",positive,0.400000,0.720000


# Considerations

## Missclassification

The dataset comes with default sentiments, but the function didn't classify them perfectly

In [11]:
# Comments with different classes
differences = df[df["sentiment"] != df["sentiment_blob"]][["review_text", "sentiment", "sentiment_blob", "polarity", "subjectivity"]]
differences.iloc[:, :3].head(10)

,review_text,sentiment,sentiment_blob
8,"product is okay, nothing special.",neutral,positive
14,customer service was unhelpful.,negative,neutral
23,customer service was unhelpful.,negative,neutral
26,average experience overall.,neutral,negative
28,works fine but could be better.,neutral,positive
34,works fine but could be better.,neutral,positive
37,customer service was unhelpful.,negative,neutral
42,works fine but could be better.,neutral,positive
48,average experience overall.,neutral,negative
57,"product is okay, nothing special.",neutral,positive


This is because the model sometimes can't divide clearly the sentiment based on the **polarity** metric. That's why there more **objective** comments in the missclassify comments.



In [10]:
# Basic statistics from polarity and subjectivity
differences[["polarity", "subjectivity"]].describe()

,polarity,subjectivity
count,6072.000000,6072.000000
mean,0.187135,0.306392
std,0.216356,0.247236
min,-0.075000,0.000000
25%,0.000000,0.000000
50%,0.291667,0.500000
75%,0.428571,0.535714
max,0.458333,0.583333


**Important terms**

*   **Polarity**: This refers to the emotional tone of the text. It ranges from -1.0 (negative) to +1.0 (positive), with 0 being neutral.
*   **Subjectivity**: This refers to the personal opinion or feeling expressed in the text. It ranges from 0.0 (objective) to 1.0 (subjective).

## Counts of the classes

In [5]:
# Amount of each original class
df["sentiment"].value_counts()

,count
sentiment,
positive,9978
negative,9937
neutral,5085


In [6]:
# Amount of classes created
df["sentiment_blob"].value_counts()

,count
sentiment_blob,
positive,13060
negative,8913
neutral,3027


In [7]:
# Classes with differences
differences.iloc[:, 1:].value_counts()

,,count
sentiment,sentiment_blob,
neutral,positive,3082
negative,neutral,2007
neutral,negative,983


As we can see in the different counts, the comments which our model have problems are with neutrals and negative classes. Even though this doesn't have much weight with the real classifications. This can be prove in the next cell.

In [8]:
print(f"Percentage of wrong classes: {(differences.shape[0] / df.shape[0]):.2%}")

Percentage of wrong classes: 24.29%


In general, the model can identify most of the comments and their main message very quickly (no matter the field).

# Data exportation

In [17]:
# Columns to export
columns = ["review_text", "sentiment_blob", "polarity", "subjectivity"]

# Export to JSON
df[columns].to_json("analyzed_sentiments.json")